# Transformer Forecaster — Versión Lite


## 1. Imports

In [2]:
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os
import pandas as pd

## 2. Configuración global

In [3]:
# ── Hiperparámetros ──────────────────────────────────────────────────────────
FREQ     = 144          
PRED_LEN = FREQ * 7    
SEQ_LEN  = FREQ * 7    
                       
EPOCHS   = 20
BATCH    = 16
LR       = 1e-3
MODEL_PATH = "transformer_lite.pth"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# ── Arquitectura (ligera) ────────────────────────────────────────────────────
D_MODEL        = 16
NHEAD          = 4
ENC_LAYERS     = 1
DEC_LAYERS     = 1
DIM_FEEDFORWARD = 64
DROPOUT        = 0.1

print(f"Device: {DEVICE}")
print(f"SEQ_LEN={SEQ_LEN} | PRED_LEN={PRED_LEN} | BATCH={BATCH}")

Device: cpu
SEQ_LEN=1008 | PRED_LEN=1008 | BATCH=16


## 3. Carga y normalización de datos

In [4]:
data = pd.read_csv("../powerconsumption.csv")
data["Datetime"] = pd.to_datetime(data["Datetime"])
data = data.sort_values("Datetime").set_index("Datetime")

series = data["PowerConsumption_Zone1"].dropna().values.astype("float32")

# ── Normalización Min-Max al rango [0, 1] ────────────────────────────────────
# IMPORTANTE: fit SOLO sobre train para no filtrar información del futuro.
split = int(len(series) * 0.8)

s_min = series[:split].min()
s_max = series[:split].max()

series_norm = (series - s_min) / (s_max - s_min)

train_series = series_norm[:split]
test_series  = series_norm[split:]

print(f"Train: {len(train_series):,} muestras | Test: {len(test_series):,} muestras")
print(f"Rango normalizado — min: {series_norm.min():.3f}, max: {series_norm.max():.3f}")

Train: 41,932 muestras | Test: 10,484 muestras
Rango normalizado — min: 0.000, max: 1.000


## 4. Dataset y DataLoader

In [5]:
class TimeSeriesDataset(Dataset):
    """Ventana deslizante sobre una serie temporal 1-D."""

    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data     = data
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        # Shape: (seq_len, 1) y (pred_len, 1)
        return (
            torch.tensor(x, dtype=torch.float32).unsqueeze(-1),
            torch.tensor(y, dtype=torch.float32).unsqueeze(-1),
        )


train_ds = TimeSeriesDataset(train_series, SEQ_LEN, PRED_LEN)
test_ds  = TimeSeriesDataset(test_series,  SEQ_LEN, PRED_LEN)

# pin_memory=True acelera la transferencia CPU→GPU cuando hay CUDA disponible
_pin = DEVICE == "cuda"
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=0, pin_memory=_pin)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                      num_workers=0, pin_memory=_pin)

print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

Train batches: 2495 | Test batches: 530


## 5. Modelo Transformer

In [6]:
class PositionalEncoding(nn.Module):
    """Encoding sinusoidal estándar (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe       = torch.zeros(max_len, d_model)           # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))        # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        return self.dropout(x + self.pe[:, : x.size(1)])

In [7]:
class TransformerForecasterLite(nn.Module):
    """
    Transformer encoder-decoder para predicción de series temporales.
    Versión Lite: d_model=16, 1 capa enc/dec, feedforward=64.
    """

    def __init__(
        self,
        pred_len: int,
        seq_len: int,
        input_size: int = 1,
        d_model: int = D_MODEL,
        nhead: int = NHEAD,
        num_encoder_layers: int = ENC_LAYERS,
        num_decoder_layers: int = DEC_LAYERS,
        dim_feedforward: int = DIM_FEEDFORWARD,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.pred_len = pred_len
        self.d_model  = d_model

        # Proyecciones de entrada
        self.src_proj = nn.Linear(input_size, d_model)
        self.tgt_proj = nn.Linear(input_size, d_model)

        # Positional encoding (max_len = ventana más grande)
        self.pos_enc = PositionalEncoding(
            d_model, max_len=max(seq_len, pred_len) + 1, dropout=dropout
        )

        # Núcleo Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,   # (B, T, d_model) en lugar de (T, B, d_model)
        )

        # Proyección de salida
        self.out_proj = nn.Linear(d_model, input_size)

    def forward(
        self,
        src: torch.Tensor,   # (B, seq_len, input_size)
        tgt: torch.Tensor,   # (B, pred_len, input_size)
    ) -> torch.Tensor:       # (B, pred_len, input_size)

        src = self.pos_enc(self.src_proj(src))   # (B, seq_len, d_model)
        tgt = self.pos_enc(self.tgt_proj(tgt))   # (B, pred_len, d_model)

        # Máscara causal: el decoder no puede mirar posiciones futuras
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1), device=src.device
        )

        out = self.transformer(src, tgt, tgt_mask=tgt_mask)  # (B, pred_len, d_model)
        return self.out_proj(out)                             # (B, pred_len, input_size)


# Instanciar y mostrar parámetros
model = TransformerForecasterLite(pred_len=PRED_LEN, seq_len=SEQ_LEN).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {total_params:,}")

Parámetros entrenables: 7,825


## 6. Entrenamiento

In [8]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
) -> float:
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)  # (B, seq_len, 1), (B, pred_len, 1)

        # Teacher forcing:
        #   entrada del decoder = [último valor del encoder] + [y sin el último paso]
        tgt_input = torch.cat([x[:, -1:, :], y[:, :-1, :]], dim=1)  # (B, pred_len, 1)

        optimizer.zero_grad()
        pred = model(x, tgt_input)           # (B, pred_len, 1)
        loss = criterion(pred, y)
        loss.backward()

        # Gradient clipping: evita explosión de gradientes
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> float:
    """Evalúa el modelo sobre un DataLoader completo."""
    model.eval()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        tgt_input = torch.cat([x[:, -1:, :], y[:, :-1, :]], dim=1)
        pred = model(x, tgt_input)
        total_loss += criterion(pred, y).item()

    return total_loss / len(loader)

In [ ]:
if os.path.exists(MODEL_PATH):
    print(f"Modelo encontrado en '{MODEL_PATH}', cargando pesos...")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Modelo cargado. Saltando entrenamiento.")

else:
    print("No se encontró modelo guardado. Entrenando desde cero...")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

    # CosineAnnealingLR: baja el LR suavemente hasta ~0 a lo largo de todo el
    # entrenamiento. Mejor convergencia que StepLR para pocos epochs.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )
    criterion = nn.MSELoss()

    history = {"train": [], "val": []}
    best_val = float("inf")

    for epoch in range(1, EPOCHS + 1):
        tr_loss  = train_epoch(model, train_dl, optimizer, criterion, DEVICE)
        val_loss = eval_epoch(model, test_dl,  criterion, DEVICE)
        scheduler.step()

        history["train"].append(tr_loss)
        history["val"].append(val_loss)

        # Guardar el mejor modelo (early-stopping manual)
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), MODEL_PATH)

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:3d}/{EPOCHS} "
                f"| Train MSE: {tr_loss:.6f} "
                f"| Val MSE: {val_loss:.6f} "
                f"| LR: {scheduler.get_last_lr()[0]:.2e}"
            )

    print(f"\nMejor val MSE: {best_val:.6f} → pesos guardados en '{MODEL_PATH}'")

    # Curva de pérdida
    plt.figure(figsize=(8, 3))
    plt.plot(history["train"], label="Train")
    plt.plot(history["val"],   label="Validación")
    plt.xlabel("Epoch")
    plt.ylabel("MSE (normalizado)")
    plt.title("Curva de pérdida")
    plt.legend()
    plt.tight_layout()
    plt.savefig("loss_curve.png", dpi=150)
    plt.show()

No se encontró modelo guardado. Entrenando desde cero...


## 7. Inferencia autoregresiva

In [ ]:
@torch.no_grad()
def predict_autoregressive(
    model: nn.Module,
    src: torch.Tensor,   # (1, seq_len, 1)
    pred_len: int,
    device: str,
) -> np.ndarray:         # (pred_len,)
    """
    Inferencia autoregresiva paso a paso.
    El decoder arranca con el último valor conocido y va
    realimentándose con sus propias predicciones.
    """
    model.eval()
    src = src.to(device)

    # Token de inicio: último valor de la ventana histórica
    tgt = src[:, -1:, :]    # (1, 1, 1)
    preds = []

    for _ in range(pred_len):
        out      = model(src, tgt)       # (1, t, 1)
        next_val = out[:, -1:, :]        # (1, 1, 1) — último paso predicho
        preds.append(next_val)
        tgt = torch.cat([tgt, next_val], dim=1)  # extiende el decoder

    return torch.cat(preds, dim=1).squeeze().cpu().numpy()  # (pred_len,)

## 8. Evaluación visual y métricas

In [ ]:
# Cargar los mejores pesos antes de evaluar
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

# Primer ejemplo del conjunto de test
x_test, y_test = test_ds[0]
src   = x_test.unsqueeze(0)               # (1, seq_len, 1)
preds = predict_autoregressive(model, src, PRED_LEN, DEVICE)  # (pred_len,)

# ── Desnormalizar para métricas en escala original ───────────────────────────
def denorm(x_norm):
    return x_norm * (s_max - s_min) + s_min

y_true_raw = denorm(y_test.squeeze().numpy())
y_pred_raw = denorm(preds)

mae  = np.mean(np.abs(y_true_raw - y_pred_raw))
rmse = np.sqrt(np.mean((y_true_raw - y_pred_raw) ** 2))
mape = np.mean(np.abs((y_true_raw - y_pred_raw) / (y_true_raw + 1e-8))) * 100

print(f"MAE  : {mae:.2f} W")
print(f"RMSE : {rmse:.2f} W")
print(f"MAPE : {mape:.2f} %")

# ── Gráfica ──────────────────────────────────────────────────────────────────
t_hist = np.arange(SEQ_LEN)
t_pred = np.arange(SEQ_LEN, SEQ_LEN + PRED_LEN)

plt.figure(figsize=(14, 4))
plt.plot(t_hist, denorm(x_test.squeeze().numpy()),
         color="steelblue", label="Historia", linewidth=0.8)
plt.plot(t_pred, y_true_raw,
         color="green", linestyle="--", label="Real", linewidth=1.2)
plt.plot(t_pred, y_pred_raw,
         color="tomato", linestyle=":", label="Predicción", linewidth=1.4)
plt.axvline(SEQ_LEN, color="gray", linestyle="-", linewidth=0.8, alpha=0.6)
plt.xlabel("Paso de tiempo (10 min)")
plt.ylabel("Consumo eléctrico (W)")
plt.title(f"Transformer Lite — Predicción 1 semana | MAE={mae:.1f} W  RMSE={rmse:.1f} W")
plt.legend()
plt.tight_layout()
plt.savefig("forecast_lite.png", dpi=150)
plt.show()
print("Gráfica guardada en 'forecast_lite.png'")